# DemoSpeedup, step by step

A guided run of the port in `ur10_clearpath/Yunfei/crisp_gym/demospeedup/` —
[DemoSpeedup: Accelerating Visuomotor Policies via Entropy-Guided Demonstration
Acceleration](https://arxiv.org/abs/2506.05064) (Guo, Xue, Xu, Xu — CoRL 2025),
[upstream code](https://github.com/lingxiao-guo/DemoSpeedup).

### The idea in one paragraph

A demonstration is not uniformly hard. Where the policy is **confident** about what
comes next, the demo can be replayed faster without losing anything; where it is
**uncertain** — around contacts, grasps, insertions — it must not be. DemoSpeedup
measures that confidence with the policy itself: sample many action chunks from a
trained ACT's CVAE prior, and take the entropy of the resulting cloud. High entropy
means many equally-good options, i.e. free motion, i.e. safe to rush.

### The four steps

| step | what | where |
| --- | --- | --- |
| 1 | train a **proxy** ACT on the original demos | `train_proxy_act.py` — *usually skippable* |
| 2 | **label** every frame with its action entropy → precision / non-precision | `label_entropy.py` |
| 3 | **retime**: keep 1 frame in 2 (precision) or 1 in 4 (non-precision) | `convert_lerobot_to_speedup.py` |
| 4 | **train** on the accelerated dataset | `train_speedup_act.py` |

The accelerated policy emits waypoints spaced 2–4 source frames apart. Run at the
**unchanged** control rate, that *is* the speedup — nothing about the runtime changes.

### How this notebook runs

The heavy steps are shelled out to `conda run -n lerobot-041 …` so they use the same
environment as every other training run in this repo, whichever kernel you started.
The notebook itself only needs `numpy`, `pandas`, `av` and (for the plots)
`matplotlib` — pick the `lerobot` kernel if you want the figures.

It works on a **handful of episodes** by default so a full pass takes minutes, not
hours. Everything it writes is listed in the last cell.


## 0. Setup

`label_entropy.py` writes its labels as a sidecar *inside the source dataset*
(`meta/demospeedup/`); the recorded parquet and mp4 files are never touched. The
cleanup cell at the end removes it again.


In [ ]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# --- what to run on -------------------------------------------------------
DEMOSPEEDUP = Path("/home/batur/Coding/ur10_clearpath/Yunfei/crisp_gym/demospeedup")
DATASET = Path("/home/batur/Coding/data/merged_act_finetune_20260528")
PROXY = Path(
    "/home/batur/Coding/ur10_clearpath/Yunfei/crisp_gym/outputs/train/"
    "act_cart7_v2_angleaxis_nogrip_chunk100_ft_20260528/checkpoints/last/pretrained_model"
)
ACCEL = Path("/home/batur/Coding/data/demospeedup_notebook_demo")  # written by step 3

ENV = "lerobot-041"   # conda env the pipeline steps run in
EPISODES = 3          # keep small; the full 70-episode pass is ~25 min
# --------------------------------------------------------------------------

sys.path.insert(0, str(DEMOSPEEDUP))

for label, path in [("demospeedup", DEMOSPEEDUP), ("dataset", DATASET), ("proxy ckpt", PROXY)]:
    print(f"{label:12s} {'OK ' if path.exists() else 'MISSING'} {path}")

try:
    import matplotlib.pyplot as plt

    PLOTS = True
except ImportError:  # lerobot-041 has no matplotlib; the tables still work
    PLOTS = False
    print("\nmatplotlib not importable in this kernel -- figures will be skipped."
          "\nUse the 'lerobot' kernel (or pip install matplotlib) to see them.")


In [ ]:
def run(args, env=ENV, cwd=DEMOSPEEDUP, quiet_prefixes=("Svt[", "Encoder ", "SVT-")):
    """Run a pipeline step in `env` and stream its output into the notebook.

    subprocess output written to an inherited fd does not show up in a cell, so
    the lines are read and re-printed here. Video-encoder chatter is dropped.
    """
    cmd = ["conda", "run", "--no-capture-output", "-n", env, "python", *map(str, args)]
    print("$ " + " ".join(cmd[5:]) + "\n")
    proc = subprocess.Popen(
        cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    lines = []
    for line in proc.stdout:
        lines.append(line)
        stripped = line.rstrip()
        if stripped.startswith(quiet_prefixes) or "Warning" in stripped:
            continue
        print(stripped)
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"step failed with exit code {code}")
    return "".join(lines)


## 1. Does the port still work?

46 tests over the vendored pieces: the entropy estimators, the CVAE-prior latent
patch, the temporal-aggregation buffer, the segmentation rules, and the stride walk
(including a parity check against upstream's own index walk). Run these before
trusting anything below.


In [ ]:
run(["-m", "pytest", "tests/", "-q"])


## 2. Pick the proxy policy

Upstream trains a proxy ACT purely to use it as an entropy oracle. **Any ACT
checkpoint already trained on this dataset is a valid proxy** — no need to train
another one. What matters is that its features match the dataset it will label.


In [ ]:
info = json.loads((DATASET / "meta" / "info.json").read_text())
cfg = json.loads((PROXY / "config.json").read_text())

print(f"dataset : {info['total_episodes']} episodes, {info['total_frames']} frames "
      f"@ {info['fps']} fps")
print(f"proxy   : chunk_size={cfg['chunk_size']}  latent_dim={cfg['latent_dim']}  "
      f"use_vae={cfg['use_vae']}\n")

rows = []
for key, feat in {**cfg["input_features"], **cfg["output_features"]}.items():
    have = info["features"].get(key)
    # LeRobot stores images HWC on disk and CHW in the policy config.
    shape = tuple(have["shape"]) if have else None
    if have and have["dtype"] == "video":
        shape = (shape[2], shape[0], shape[1])
    rows.append({
        "feature": key,
        "policy": tuple(feat["shape"]),
        "dataset": shape,
        "match": shape == tuple(feat["shape"]),
    })
compat = pd.DataFrame(rows)
display(compat)
assert compat["match"].all(), "proxy checkpoint does not match this dataset"
print("\nProxy is compatible -- step 1 (train_proxy_act.py) can be skipped.")


If you *did* need a fresh proxy, this is the command — a plain `lerobot-train` run
with the same augmentation as the ACT baselines in `examples/28` and `examples/30`.
`--print-only` shows it without launching a 30k-step job.


In [ ]:
run(["train_proxy_act.py", "--print-only",
     f"--dataset-root={DATASET}", f"--repo-id={DATASET.name}"])


## 3. Step 2 — label the demonstrations

For every frame of every episode:

1. draw `--num-samples 10` action chunks from the CVAE prior (`z ~ N(0, I)`),
2. pool the samples of **every still-valid chunk covering this frame** — with
   `chunk_size = 100` that is up to 100 predictions × 10 draws = 1000 samples of
   the action *now* (upstream's temporal aggregation),
3. take the KDE differential entropy of that cloud,
4. z-normalise the trace per episode and split it into precision / non-precision.

Cost is ~65 ms/frame on a 4090. Three episodes ≈ 40 s; all 70 ≈ 25 min.


In [ ]:
labels_path = DATASET / "meta" / "demospeedup" / "labels.parquet"
if labels_path.exists():
    print(f"labels already present at {labels_path} -- delete it to re-label")
else:
    run(["label_entropy.py",
         f"--dataset-root={DATASET}",
         f"--policy-path={PROXY}",
         f"--episodes={EPISODES}",
         "--frames-per-forward=2",
         "--num-workers=2"])


### What landed on disk

`meta/demospeedup/labels.parquet` — one row per frame — and
`meta/demospeedup.json`, which records exactly how those labels were produced
(checkpoint, sample count, KDE bandwidth, segmenter, strides, seed) so a later
conversion can be traced back to it.


In [ ]:
from lerobot_bridge import read_labels

frame_labels, label_config = read_labels(DATASET)
display(frame_labels.head())

summary = pd.DataFrame(label_config["per_episode"])
display(summary)
print(json.dumps({k: v for k, v in label_config.items()
                  if k not in ("per_episode", "labelled_at_s")}, indent=2))


## 4. What the entropy trace looks like

This is the whole method in one picture. The shaded stretches are where the policy
was confident (high entropy → many equally good continuations → free motion); those
get the 4× stride. The valleys are where it was not, and those keep the 2× stride.


In [ ]:
EP = int(frame_labels["episode_index"].iloc[0])
ep_rows = frame_labels[frame_labels.episode_index == EP].sort_values("frame_index")
entropy = ep_rows["entropy"].to_numpy()
labels = ep_rows["label"].to_numpy()

if PLOTS:
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(entropy, color="tab:blue", lw=1.2, label="KDE entropy")
    ax.axhline(entropy.mean(), color="grey", ls="--", lw=1, label="episode mean")
    ax.fill_between(np.arange(len(labels)), entropy.min(), entropy.max(),
                    where=labels == 1, color="tab:red", alpha=0.15,
                    label="non-precision (4x stride)")
    ax.set(xlabel="source frame", ylabel="entropy", title=f"episode {EP}")
    ax.legend(loc="lower right", fontsize="small")
    fig.tight_layout()
else:
    print(f"episode {EP}: {len(entropy)} frames, "
          f"entropy {entropy.min():.2f}..{entropy.max():.2f}, "
          f"{100 * labels.mean():.0f}% non-precision")


## 5. From labels to kept frames

`select_keep_indices` walks the episode with DemoSpeedup's variable stride and
returns the frames the accelerated episode keeps. Everything else is dropped —
parquet rows and video frames alike.

The tick marks below are the survivors. Notice they bunch up in the precision
valleys and thin out through the confident stretches: that is the entire
contribution of the method.


In [ ]:
from demospeedup_core.retiming import retiming_stats, select_keep_indices

keep = select_keep_indices(labels)
stats = retiming_stats(labels, keep)
print(f"episode {EP}: {stats.n_source} -> {stats.n_kept} frames "
      f"({stats.speedup:.2f}x), largest gap {stats.max_gap} source frames")

if PLOTS:
    fig, ax = plt.subplots(figsize=(11, 1.9))
    ax.fill_between(np.arange(len(labels)), 0, 1, where=labels == 1,
                    color="tab:red", alpha=0.15, transform=ax.get_xaxis_transform())
    ax.vlines(keep, 0.25, 0.75, color="tab:green", lw=1.1)
    ax.set(xlabel="source frame", yticks=[], xlim=(0, len(labels)),
           title=f"kept frames — {stats.n_kept} of {stats.n_source} ({stats.speedup:.2f}x)")
    fig.tight_layout()


## 6. Choosing the strides — the safety check

`(low_v, high_v) = (2, 4)` is upstream's setting, tuned on a simulated Aloha. On a
real UR10e what matters is **how far the end-effector is commanded to move per
control period** once the in-between frames are gone. At 20 Hz a 4× stride through
a fast free-space phase turns a 25 mm/frame motion into 100 mm per step — 2 m/s at
the tool.

Sweep the strides and look at the last column before committing.


In [ ]:
from lerobot_bridge import episode_actions, episode_ranges, labels_by_episode

by_episode = labels_by_episode(frame_labels)
episodes = [e for e in episode_ranges(DATASET) if e.episode_index in by_episode]
actions = {e.episode_index: episode_actions(DATASET, e, info) for e in episodes}
fps = float(info["fps"])


def sweep(low_v, high_v):
    kept = total = 0
    worst_step = 0.0
    for ep in episodes:
        lab = by_episode[ep.episode_index]
        k = select_keep_indices(lab, low_v, high_v)
        steps = np.linalg.norm(np.diff(actions[ep.episode_index][k, :3], axis=0), axis=-1)
        worst_step = max(worst_step, float(steps.max()))
        kept += len(k)
        total += len(lab)
    return {
        "low_v": low_v,
        "high_v": high_v,
        "frames": f"{total} -> {kept}",
        "speedup": round(total / kept, 2),
        "worst step (mm)": round(worst_step * 1000, 1),
        "worst speed (m/s)": round(worst_step * fps, 2),
    }


display(pd.DataFrame([sweep(1, 1), sweep(2, 3), sweep(2, 4), sweep(3, 6)]))
print("(1, 1) is the untouched dataset -- the baseline row for the two right columns.")


### Segmenter backends

Upstream clusters `(z(frame_index), z(entropy))` with HDBSCAN. Neither `scikit-learn`
nor `hdbscan` is installed in `lerobot-041`, so the default backend is `threshold`:
split at the episode mean, enforce a minimum run length, then apply upstream's own
cluster rule to the merged runs. On a feature pair that is monotone in time those
clusters *are* essentially those runs — this cell checks how close the two agree on
your data, re-segmenting the **stored** entropy so the policy never runs again.


In [ ]:
from demospeedup_core.segmentation import segment_entropy

try:
    import sklearn  # noqa: F401

    rows = []
    for ep in episodes:
        e = frame_labels[frame_labels.episode_index == ep.episode_index] \
            .sort_values("frame_index")["entropy"].to_numpy()
        a = segment_entropy(e, backend="threshold").labels
        b = segment_entropy(e, backend="hdbscan").labels
        rows.append({
            "episode": ep.episode_index,
            "threshold fast %": round(100 * a.mean(), 1),
            "hdbscan fast %": round(100 * b.mean(), 1),
            "frames disagreeing %": round(100 * float((a != b).mean()), 1),
        })
    display(pd.DataFrame(rows))
except ImportError:
    print("scikit-learn is not in this kernel -- run this cell in the 'lerobot' env "
          "to compare backends. The pipeline default ('threshold') needs numpy only.")


## 7. Step 3 — write the accelerated dataset

Parquet rows for dropped frames are removed and every camera stream is re-encoded
without them. What comes out is a stock LeRobot v3.0 dataset — `lerobot-train`
needs no patch to consume it, and the existing crisp_gym replay tooling can play it
back to see what the policy will be asked to imitate.

The frame *rate* is unchanged. A retimed episode holds the same motion in 2–4×
fewer frames, so replaying it at the source fps is what makes the robot move faster.


In [ ]:
if ACCEL.exists():
    print(f"{ACCEL} already exists -- delete it to re-convert")
else:
    run(["convert_lerobot_to_speedup.py",
         f"--src={DATASET}", f"--dst={ACCEL}", f"--limit-episodes={EPISODES}"])


### Verify the conversion

Three things have to hold, and none of them are obvious enough to take on trust:
the surviving actions must be the source actions at the kept indices, the new
timeline must be contiguous at the original fps, and — the one that silently breaks
frame/row alignment if the video reader drifts — the surviving **video frames** must
be the source frames at those same indices.

The last check compares each kept frame against its source frame, and against a
deliberately shifted one as a control. Matched frames differ only by AV1 re-encode
loss; the control should differ far more.


In [ ]:
import av

new_df = pd.concat([pd.read_parquet(p) for p in sorted((ACCEL / "data").rglob("*.parquet"))])
src_df = pd.concat([pd.read_parquet(p) for p in sorted((DATASET / "data").rglob("*.parquet"))])

ep_new = new_df[new_df.episode_index == EP].sort_values("frame_index")
ep_src = src_df[src_df.episode_index == EP].sort_values("frame_index")
src_actions = np.stack(ep_src["action"].to_numpy())
new_actions = np.stack(ep_new["action"].to_numpy())

print("rows kept          :", len(new_actions), "of", len(src_actions))
print("actions match      :", np.allclose(src_actions[keep], new_actions))
print("timeline contiguous:", np.allclose(ep_new["timestamp"].to_numpy(),
                                          np.arange(len(new_actions)) / fps))


def decode(path, wanted):
    """Decode the frames at `wanted` (sorted absolute indices) from an mp4."""
    wanted, out = sorted(wanted), {}
    with av.open(str(path)) as container:
        for i, frame in enumerate(container.decode(video=0)):
            if i in wanted:
                out[i] = frame.to_ndarray(format="rgb24")
            if i >= wanted[-1]:
                break
    return out


VIDEO_KEY = "observation.images.d405"  # the wrist camera moves most -> strictest check
tmpl = info["video_path"]
src_video = DATASET / tmpl.format(video_key=VIDEO_KEY, chunk_index=0, file_index=0)
new_video = ACCEL / tmpl.format(video_key=VIDEO_KEY, chunk_index=0, file_index=0)

probe = [0, len(keep) // 3, len(keep) // 2, len(keep) - 1]
new_frames = decode(new_video, probe)
src_frames = decode(src_video, [int(keep[j]) for j in probe] + [int(keep[len(keep) // 2]) + 3])

matched = [np.abs(new_frames[j] / 255 - src_frames[int(keep[j])] / 255).mean() for j in probe]
control = np.abs(new_frames[len(keep) // 2] / 255
                 - src_frames[int(keep[len(keep) // 2]) + 3] / 255).mean()
print(f"\nframe error at kept indices : {[round(m, 4) for m in matched]}")
print(f"control (source frame + 3)  : {control:.4f}   <- must be clearly larger")


In [ ]:
if PLOTS:
    fig, axes = plt.subplots(2, 4, figsize=(13, 5))
    for col, j in enumerate(probe):
        axes[0, col].imshow(src_frames[int(keep[j])])
        axes[0, col].set_title(f"source frame {int(keep[j])}", fontsize=9)
        axes[1, col].imshow(new_frames[j])
        axes[1, col].set_title(f"accelerated frame {j}", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    axes[0, 0].set_ylabel("source")
    fig.suptitle(f"{VIDEO_KEY} — same moments, {stats.speedup:.2f}x fewer frames "
                 f"between them", fontsize=11)
    fig.tight_layout()


### What the arm actually has to do

Same motion, fewer control periods. The distribution on the right is what the CRISP
cartesian controller will be asked to track — if its tail runs past what the
hardware follows comfortably, come back to §6 and lower `--high-v`.


In [ ]:
src_speed = np.linalg.norm(np.diff(src_actions[:, :3], axis=0), axis=-1) * fps
new_speed = np.linalg.norm(np.diff(new_actions[:, :3], axis=0), axis=-1) * fps

print(f"source      : mean {src_speed.mean():.3f} m/s, max {src_speed.max():.3f} m/s")
print(f"accelerated : mean {new_speed.mean():.3f} m/s, max {new_speed.max():.3f} m/s")

if PLOTS:
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.4))
    left.plot(np.arange(len(src_speed)) / fps, src_speed, lw=1, color="tab:blue")
    left.plot(np.arange(len(new_speed)) / fps, new_speed, lw=1, color="tab:red")
    left.set(xlabel="seconds of replay", ylabel="commanded speed (m/s)",
             title="the episode gets shorter")
    left.legend(["source", "accelerated"], fontsize="small")
    right.hist([src_speed, new_speed], bins=30, label=["source", "accelerated"],
               color=["tab:blue", "tab:red"])
    right.set(xlabel="commanded speed (m/s)", ylabel="control periods",
              title="per-step demand")
    right.legend(fontsize="small")
    fig.tight_layout()


## 8. Step 4 — train on it

The only differences from the proxy run are the dataset and the chunk size. Upstream
halves it — 50 → 25 for ACT — "to maintain geometrical consistency": one accelerated
frame covers 2–4 source frames, so half the chunk length still spans about as much
*motion*. Our ACT baselines use `chunk_size=100`, hence 50.

The training script refuses to start unless the dataset carries the
`meta/demospeedup_source.json` sidecar, so a baseline dataset cannot end up in an
accelerated run by accident.


In [ ]:
run(["train_speedup_act.py", "--print-only",
     f"--dataset-root={ACCEL}", f"--repo-id={ACCEL.name}",
     "--output-dir=outputs/train/demospeedup_act_demo"])


Drop `--print-only` (and add `--wandb`) to actually launch it. Then evaluate the
accelerated policy exactly like any other ACT checkpoint in this repo — **at the
unchanged control rate**. The speedup is baked into the action deltas; raising the
replay fps on top of it applies the acceleration twice.


## 9. Cleanup

This notebook wrote:

- `<DATASET>/meta/demospeedup/labels.parquet` + `<DATASET>/meta/demospeedup.json` — the label
  sidecar. A partial one (3 of 70 episodes) is worth removing so a later full run
  starts clean; `convert_lerobot_to_speedup.py` refuses to run against a sidecar
  that does not cover every episode anyway.
- `<ACCEL>` (`/home/batur/Coding/data/demospeedup_notebook_demo`) — the
  3-episode demo dataset.

Set `CLEAN = True` to remove both.


In [ ]:
CLEAN = False

if CLEAN:
    for path in [DATASET / "meta" / "demospeedup", DATASET / "meta" / "demospeedup.json", ACCEL]:
        if path.is_dir():
            shutil.rmtree(path)
            print("removed", path)
        elif path.exists():
            path.unlink()
            print("removed", path)
else:
    print("CLEAN = False -- nothing removed. Left behind:")
    for path in [DATASET / "meta" / "demospeedup", ACCEL]:
        print(f"  {'exists ' if path.exists() else 'absent '} {path}")
